In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.stats import spearmanr
from scipy.stats import linregress
import statsmodels.formula.api as smf

# Construct mobility network using mobile phone data

In [ ]:
# We load Oxfordshire MSOA boundary data that only contains Local Authority Districts (LADs): Cherwell, Oxford, South Oxfordshire, Vale of White Horse, West Oxfordshire
msoa_ox_gdf = gpd.read_file('/vsigzip//vsicurl/https://raw.githubusercontent.com/jishan96/OSSEN_MPD_bias/main/data/oxfordshire_msoa.geojson.gz')
msoa_ox_gdf = msoa_ox_gdf.to_crs(epsg=27700)
print(f'Oxfordshire contains MSOAs: {msoa_ox_gdf.shape[0]}')
# We get list of MSOA IDs in Oxfordshire
ox_msoa_ids = msoa_ox_gdf['MSOA21CD'].tolist()
print(f'Oxfordshire MSOA IDs: {ox_msoa_ids[:5]} ...')
msoa_ox_gdf.head()

In [ ]:
# We load mobile phone data
mpd_df = pd.read_csv('https://raw.githubusercontent.com/jishan96/OSSEN_MPD_bias/main/data/msoa_OD_travel2work.csv.gz', compression='gzip')
mpd_df.head()

In [ ]:
# We filter to Oxfordshire MSOAs OD pairs
mpd_ox_od = mpd_df[(mpd_df['MSOA21CD_home'].isin(ox_msoa_ids)) & (mpd_df['MSOA21CD_work'].isin(ox_msoa_ids))]
print(f'Oxfordshire OD pairs: {mpd_ox_od.shape[0]}')
mpd_ox_od.head()

In [ ]:
# We construct OD commuting network based on mobile phone data
# We create a directed network
G_mpd = nx.DiGraph()
# We add nodes, each node represents an MSOA in Oxfordshire, with attributes for latitude, longitude, and name
for _, row in msoa_ox_gdf.iterrows():
    G_mpd.add_node(row['MSOA21CD'], lat=row.geometry.centroid.y, lon=row.geometry.centroid.x, name=row['MSOA21NM'])
pos_mpd = {node: (data['lon'], data['lat']) for node, data in G_mpd.nodes(data=True)}
labels_mpd = {node: data['name'] for node, data in G_mpd.nodes(data=True)}
nodelist = list(G_mpd.nodes())
print(f'Number of nodes: {G_mpd.number_of_nodes()}')

In [ ]:
# We add directed edges with weights representing the number of commuters from home MSOA to work MSOA
for _, row in mpd_ox_od.iterrows():
    G_mpd.add_edge(row['MSOA21CD_home'], row['MSOA21CD_work'], count=row['count'])
print(f'Number of edges: {G_mpd.number_of_edges()}')

In [ ]:
# We plot the OD commuting network based on mobile phone data
fig, ax = plt.subplots(figsize=(8,8))
edges = [(u, v) for u, v in G_mpd.edges() if u != v]
counts = [G_mpd[u][v]['count'] for u, v in edges]
widths = [np.log1p(c) / np.log1p(max(counts)) * 4 for c in counts]

msoa_ox_gdf.plot(ax=ax, color='white', edgecolor='black', linewidth=0.5)
nx.draw_networkx_nodes(G_mpd, pos=pos_mpd, ax=ax, node_size=40, node_color='black')
nx.draw_networkx_edges(G_mpd, pos=pos_mpd, ax=ax, edgelist=edges, width=widths, arrows=True,
                       arrowsize=8, edge_color='lightgray', alpha=0.6,
                       connectionstyle='arc3,rad=0.2', node_size=40,
                       min_source_margin=0, min_target_margin=0)
ax.set_title('OD Network based on Mobile Phone Data')
ax.set_axis_on()
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
plt.tight_layout()
plt.show()

In [ ]:
# We compute density and average degree.
density_mpd = nx.density(G_mpd) # in directed networks, density equals to the number of edges divided by the maximum possible number of edges, which is n*(n-1) for n nodes.
avg_degree_mpd = sum(dict(G_mpd.degree()).values()) / G_mpd.number_of_nodes() # average degree equals to the sum of in and out degrees divided by the number of nodes
print(f'Mobile phone network density: {density_mpd:.2f}')
print(f'Mobile phone network average degree: {avg_degree_mpd:.2f}')

In [ ]:
# We compute in-degree and out-degree for each node.
in_deg_mpd = {n: G_mpd.in_degree(n) for n in nodelist}
out_deg_mpd = {n: G_mpd.out_degree(n) for n in nodelist}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8), gridspec_kw={'width_ratios': [1, 2]})
# We plot in-degree histogram
bins = np.linspace(0, max(max(in_deg_mpd.values()), max(out_deg_mpd.values())) + 1, 30)
axes[0].hist(list(in_deg_mpd.values()), bins=bins, alpha=0.85, color='salmon')
axes[0].set_box_aspect(1)
axes[0].set_xlabel('In-degree')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('In-degree across MSOAs')
# We plot in-degree in the OD network
msoa_ox_gdf.plot(ax=axes[1], color='white', edgecolor='black', linewidth=0.5)
nodes = nx.draw_networkx_nodes(G_mpd, pos=pos_mpd, ax=axes[1],
                               nodelist=nodelist, node_size=[2 + in_deg_mpd[n] for n in nodelist],
                               node_color=[in_deg_mpd[n] for n in nodelist], cmap='YlOrRd')

nx.draw_networkx_edges(G_mpd, pos=pos_mpd, ax=axes[1], edgelist=edges, width=0.3,
                       arrows=True, arrowsize=8, edge_color='lightgray', alpha=0.6,
                       connectionstyle='arc3,rad=0.2', node_size=[2 + in_deg_mpd[n] for n in nodelist],
                       min_source_margin=0, min_target_margin=3)

fig.colorbar(nodes, ax=axes[1], label='In-degree', shrink=0.7)
axes[1].set_title('OD Network based on Mobile Phone Data (nodes colored by in-degree)')
axes[1].set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8), gridspec_kw={'width_ratios': [1, 2]})
# We plot out-degree histogram
bins = np.linspace(0, max(max(in_deg_mpd.values()), max(out_deg_mpd.values())) + 1, 30)
axes[0].hist(list(out_deg_mpd.values()), bins=bins, alpha=0.85, color='salmon')
axes[0].set_box_aspect(1)
axes[0].set_xlabel('Out-degree')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('Out-degree across MSOAs')
# We plot out-degree in the OD network
msoa_ox_gdf.plot(ax=axes[1], color='white', edgecolor='black', linewidth=0.5)
nodes = nx.draw_networkx_nodes(G_mpd, pos=pos_mpd, ax=axes[1],
                               nodelist=nodelist, node_size=[2 + out_deg_mpd[n] for n in nodelist],
                               node_color=[out_deg_mpd[n] for n in nodelist], cmap='YlOrRd')

nx.draw_networkx_edges(G_mpd, pos=pos_mpd, ax=axes[1], edgelist=edges, width=0.3,
                       arrows=True, arrowsize=8, edge_color='lightgray', alpha=0.8,
                       connectionstyle='arc3,rad=0.2', node_size=[2 + in_deg_mpd[n] for n in nodelist],
                       min_source_margin=0, min_target_margin=3)

fig.colorbar(nodes, ax=axes[1], label='Out-degree', shrink=0.7)
axes[1].set_title('OD Network based on Mobile Phone Data (nodes colored by out-degree)')
axes[1].set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# We compute raw in-strength and out-strength (sum of edge weights) for each node
in_str_mpd = {n: G_mpd.in_degree(n, weight='count') for n in nodelist}
out_str_mpd = {n: G_mpd.out_degree(n, weight='count') for n in nodelist}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8), gridspec_kw={'width_ratios': [1, 2]})
# We plot in-strength histogram
bins = np.linspace(0, max(max(in_str_mpd.values()), max(out_str_mpd.values())) + 1, 30)
axes[0].hist(list(in_str_mpd.values()), bins=bins, alpha=0.85, color='salmon')
axes[0].set_box_aspect(1)
axes[0].set_xlabel('In-strength')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('In-strength across MSOAs')
# We plot in-strength in the OD network
msoa_ox_gdf.plot(ax=axes[1], color='white', edgecolor='black', linewidth=0.5)
nodes = nx.draw_networkx_nodes(G_mpd, pos=pos_mpd, ax=axes[1],
                               nodelist=nodelist, node_size=[2 + in_str_mpd[n] for n in nodelist],
                               node_color=[in_str_mpd[n] for n in nodelist], cmap='YlOrRd')

nx.draw_networkx_edges(G_mpd, pos=pos_mpd, ax=axes[1], edgelist=edges, width=0.3,
                       arrows=True, arrowsize=8, edge_color='lightgray', alpha=0.6,
                       connectionstyle='arc3,rad=0.2', node_size=[2 + in_deg_mpd[n] for n in nodelist],
                       min_source_margin=0, min_target_margin=3)

fig.colorbar(nodes, ax=axes[1], label='In-strength', shrink=0.7)
axes[1].set_title('OD Network based on Mobile Phone Data (nodes colored by in-strength)')
axes[1].set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8), gridspec_kw={'width_ratios': [1, 2]})
# We plot out-strength histogram
bins = np.linspace(0, max(max(in_str_mpd.values()), max(out_str_mpd.values())) + 1, 30)
axes[0].hist(list(out_str_mpd.values()), bins=bins, alpha=0.85, color='salmon')
axes[0].set_box_aspect(1)
axes[0].set_xlabel('Out-strength')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('Out-strength across MSOAs')
# We plot out-strength in the OD network
msoa_ox_gdf.plot(ax=axes[1], color='white', edgecolor='black', linewidth=0.5)
nodes = nx.draw_networkx_nodes(G_mpd, pos=pos_mpd, ax=axes[1],
                               nodelist=nodelist, node_size=[2 + out_str_mpd[n] for n in nodelist],
                               node_color=[out_str_mpd[n] for n in nodelist], cmap='YlOrRd')

nx.draw_networkx_edges(G_mpd, pos=pos_mpd, ax=axes[1], edgelist=edges, width=0.3,
                       arrows=True, arrowsize=8, edge_color='lightgray', alpha=0.6,
                       connectionstyle='arc3,rad=0.2', node_size=[2 + in_deg_mpd[n] for n in nodelist],
                       min_source_margin=0, min_target_margin=3)

fig.colorbar(nodes, ax=axes[1], label='Out-strength', shrink=0.7)
axes[1].set_title('OD Network based on Mobile Phone Data (nodes colored by out-strength)')
axes[1].set_axis_off()
plt.tight_layout()
plt.show()

# Biases in mobile phone data

We compute the population coverage bias measure proposed by Cabrera and Rowe (2025) to assess the representation of the mobile phone data (MPD) across MSOAs.
Cabrera, C., & Rowe, F. (2025). A systematic machine learning approach to measure and assess biases in mobile phone population data. arXiv preprint arXiv:2509.02603.

## 1. Measuring population coverage bias

Population coverage in MSOA is calculated as:

$$c_i = \frac{P^{D}_i}{P_i} \times 100$$

Population coverage bias is then calculated as:

$$e_i = 100 - c_i$$

In [ ]:
# We load population census data for MSOAs.
pop_df = pd.read_csv('https://raw.githubusercontent.com/jishan96/OSSEN_MPD_bias/main/data/MSOA_Age_5yrG.csv.gz', compression='gzip')
pop_df.head()

In [ ]:
# We convert all columns except the first two identifier columns to numeric
pop_df.iloc[:, 2:] = pop_df.iloc[:, 2:].apply(
    pd.to_numeric, errors='coerce'
).fillna(0)
# We rename columns and create new columns for the broader age groups
pop_df = pop_df.rename(columns={
    '2021 super output area - middle layer': 'MSOA21NM',
    'mnemonic': 'MSOA21CD',
    'Total': 'pop_census',
})
pop_ox = pop_df[pop_df['MSOA21CD'].isin(ox_msoa_ids)].copy()
# We merge the population to MSOA GeoDataFrame and compute population density
msoa_ox_pop = msoa_ox_gdf.merge(pop_ox[['MSOA21CD', 'pop_census']], on='MSOA21CD', how='left')
msoa_ox_pop.head()


In [ ]:
# We compute population coverage bias in MPD.
mpd_pop = mpd_ox_od.groupby('MSOA21CD_home', as_index=False)['count'].sum()
mpd_pop = mpd_pop.rename(columns={'MSOA21CD_home': 'MSOA21CD', 'count': 'pop_mpd'})
mpd_pop = mpd_pop[mpd_pop['MSOA21CD'].isin(ox_msoa_ids)]
# We merge the MPD population with the MSOA features and compute coverage and bias
msoa_bias = msoa_ox_pop.merge(mpd_pop, on='MSOA21CD', how='left')
msoa_bias['pop_mpd'] = msoa_bias['pop_mpd'].fillna(0)
msoa_bias['coverage'] = msoa_bias['pop_mpd'] / msoa_bias['pop_census'] * 100
msoa_bias['bias'] = 100 - msoa_bias['coverage']

print('Oxfordshire bias summary:')
print(msoa_bias['bias'].describe().round(2))
msoa_bias[['MSOA21NM', 'pop_census', 'pop_mpd', 'coverage', 'bias']].sort_values('bias').head()

In [ ]:
# We plot bias of each MSOAs
# We merge the bias data to the MSOA GeoDataFrame for plotting
bias_gdf = msoa_ox_gdf.merge(msoa_bias[['MSOA21CD', 'pop_census', 'bias', 'coverage']], on='MSOA21CD', how='left')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(msoa_bias['bias'], bins=15, color='salmon', edgecolor='white', alpha=0.85)
axes[0].axvline(msoa_bias['bias'].median(), color='black', linestyle='--',
                label=f"Median = {msoa_bias['bias'].median():.1f}")
axes[0].set_xlabel('Population coverage bias ($e_i$)')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('Distribution of bias across Oxfordshire MSOAs')
axes[0].legend()

bias_gdf.plot(column='bias', ax=axes[1], legend=True, cmap='RdYlGn_r',
              edgecolor='black', linewidth=0.3,
              legend_kwds={'label': 'bias ($e_i$)', 'shrink': 0.8})
axes[1].set_title('Spatial distribution of population coverage bias')
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

## 2. Identifying sources of biases

In [ ]:
# We pick potential sources: 1. population, 2. population density, 3. age, 4. occupation class.
# We transform the coordinates to a projected CRS for area calculation (EPSG:27700) to compute density
msoa_ox_pop = msoa_ox_pop.to_crs(epsg=27700)
msoa_ox_pop['area_km2'] = msoa_ox_pop['geometry'].area / 1000000  # Convert m^2 to km^2
msoa_ox_pop['pop_density'] = msoa_ox_pop['pop_census'] / msoa_ox_pop['area_km2']
msoa_ox_pop.head()

In [ ]:
# we compute age group populations
age_ox = pop_ox.copy()
age_ox['young'] = age_ox[['Aged 15 to 19 years', 'Aged 20 to 24 years', 'Aged 25 to 29 years', 'Aged 30 to 34 years']].sum(axis=1)
age_ox['middle-aged'] = age_ox[['Aged 35 to 39 years', 'Aged 40 to 44 years', 'Aged 45 to 49 years', 'Aged 50 to 54 years', 'Aged 55 to 59 years', 'Aged 60 to 64 years']].sum(axis=1)
age_ox['older'] = age_ox[['Aged 65 to 69 years', 'Aged 70 to 74 years', 'Aged 75 to 79 years', 'Aged 80 to 84 years', 'Aged 85 years and over']].sum(axis=1)
age_ox = age_ox[['MSOA21CD', 'pop_census', 'young', 'middle-aged', 'older']]
# We compute the percentage of each age group in the population
age_ox['young_pct'] = age_ox['young'] / age_ox['pop_census'] * 100
age_ox['middle-aged_pct'] = age_ox['middle-aged'] / age_ox['pop_census'] * 100
age_ox['older_pct'] = age_ox['older'] / age_ox['pop_census'] * 100
age_ox.head()

In [ ]:

# We load occupation data for MSOAs.
occ_df = pd.read_csv('https://raw.githubusercontent.com/jishan96/OSSEN_MPD_bias/main/data/MSOA_Occupation.csv.gz', compression='gzip')
occ_df.head()

In [ ]:
# We pick occupation catagories that are likely to be wealthier: 1. Managers, directors and senior officials. We also pick occupation catagories that are likely to be less wealthy: 9. Elementary occupations.
# We convert all columns except the first two identifier columns to numeric
occ_df.iloc[:, 2:] = occ_df.iloc[:, 2:].apply(
    pd.to_numeric, errors='coerce'
).fillna(0)
# We rename columns and create new columns for the wealthier and less wealthy occupation groups
occ_df = occ_df.rename(columns={
    '2021 super output area - middle layer': 'MSOA21NM',
    'mnemonic': 'MSOA21CD',
    'Total: All usual residents aged 16 years and over in employment the week before the census': 'occ_census',
})
# We filter to Oxfordshire MSOAs
occ_ox = occ_df[occ_df['MSOA21CD'].isin(ox_msoa_ids)].copy()
occ_ox['wealthier'] = occ_ox[['1. Managers, directors and senior officials']].sum(axis=1)
occ_ox['less_wealthy'] = occ_ox[['9. Elementary occupations']].sum(axis=1)
occ_ox = occ_ox[['MSOA21CD', 'occ_census', 'wealthier', 'less_wealthy']]
# We compute the percentage of each occupation group in the population
occ_ox['wealthier_pct'] = occ_ox['wealthier'] / occ_ox['occ_census'] * 100
occ_ox['less_wealthy_pct'] = occ_ox['less_wealthy'] / occ_ox['occ_census'] * 100
occ_ox.head()

In [ ]:
# We merge age and occupation data back to msoa_ox_pop GeoDataFrame.
msoa_ox_features = msoa_ox_pop.merge(age_ox[['MSOA21CD', 'young_pct', 'middle-aged_pct', 'older_pct']], on='MSOA21CD', how='left')
msoa_ox_features = msoa_ox_features.merge(occ_ox[['MSOA21CD', 'wealthier_pct', 'less_wealthy_pct']], on='MSOA21CD', how='left')
msoa_ox_features.head()

In [ ]:
# We plot choropleth maps for sources of biases.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
map_vars = [
    ('pop_census', 'Usual resident population'),
    ('pop_density', 'Population density (residents per km²)'),
    ('young_pct', 'Young population (15-34)'),
    ('wealthier_pct', 'Wealthier population (SOC group 1)'),
]

for ax, (col, title) in zip(axes.ravel(), map_vars):
    msoa_ox_features.plot(column=col, ax=ax, legend=True, cmap='YlOrRd',
                      edgecolor='black', linewidth=0.3,
                      legend_kwds={'label': col, 'shrink': 0.7})
    ax.set_title(title)
    ax.set_axis_off()

plt.suptitle('Oxfordshire MSOAs: spatial distribution of census covariates', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Univariate exploration
# 1. we explore the relationship between bias and the young population share in that msoa
reg_df = msoa_ox_features[['MSOA21CD', 'young_pct', 'wealthier_pct', 'pop_census', 'pop_density']].merge(
    msoa_bias[['MSOA21CD', 'bias']], on='MSOA21CD', how='left'
)
reg_gdf = msoa_ox_features.merge(reg_df[['MSOA21CD', 'bias']], on='MSOA21CD', how='left')

x = reg_df['young_pct']
y = reg_df['bias']
slope_y, intercept_y, r_y, p_y, _ = linregress(x, y)


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
reg_gdf.plot(column='bias', ax=axes[0], legend=True, cmap='RdYlGn_r',
             edgecolor='black', linewidth=0.3,
             legend_kwds={'label': 'bias ($e_i$)', 'shrink': 0.8})
axes[0].set_title('Population coverage bias')
axes[0].set_axis_off()

reg_gdf.plot(column='young_pct', ax=axes[1], legend=True, cmap='YlOrRd',
             edgecolor='black', linewidth=0.3,
             legend_kwds={'label': 'young_pct (%)', 'shrink': 0.8})
axes[1].set_title('Young population share (15-34)')
axes[1].set_axis_off()
plt.suptitle('Do high-bias areas overlap with young-population areas?', y=1.02)
plt.tight_layout()
plt.show()

print('Univariate (young_pct):')
if p_y < 0.05:
    direction = 'higher' if slope_y > 0 else 'lower'
    print(f'  Significant association (p < 0.05): MSOAs with more young residents tend to have {direction} bias.')
else:
    print(f'  No statistically significant linear association (p = {p_y:.3f}).')
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(x, y, alpha=0.7, edgecolors='k', linewidths=0.3)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, intercept_y + slope_y * x_line, color='crimson', linewidth=1.5)
ax.set_xlabel('Share young (15-34, %)')
ax.set_ylabel('Bias ($e_i$)')
ax.set_title(f'Bias vs young population share (r = {r_y:.3f}, p = {p_y:.2e})')
plt.tight_layout()
plt.show()



In [ ]:
# Univariate exploration
# 2. we explore the relationship between bias and the manager occupation share in that msoa
x = reg_df['wealthier_pct']
y = reg_df['bias']
slope_w, intercept_w, r_w, p_w, _ = linregress(x, y)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
reg_gdf.plot(column='bias', ax=axes[0], legend=True, cmap='RdYlGn_r',
             edgecolor='black', linewidth=0.3,
             legend_kwds={'label': 'bias ($e_i$)', 'shrink': 0.8})
axes[0].set_title('Population coverage bias')
axes[0].set_axis_off()

reg_gdf.plot(column='wealthier_pct', ax=axes[1], legend=True, cmap='YlOrRd',
             edgecolor='black', linewidth=0.3,
             legend_kwds={'label': 'wealthier_pct (%)', 'shrink': 0.8})
axes[1].set_title('Manager occupation share (SOC group 1)')
axes[1].set_axis_off()
plt.suptitle('Do high-bias areas overlap with manager-rich areas?', y=1.02)
plt.tight_layout()
plt.show()

print('Univariate (wealthier_pct):')
if p_w < 0.05:
    direction = 'higher' if slope_w > 0 else 'lower'
    print(f'  Significant association (p < 0.05): MSOAs with more managers tend to have {direction} bias.')
else:
    print(f'  No statistically significant linear association (p = {p_w:.3f}).')

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(x, y, alpha=0.7, edgecolors='k', linewidths=0.3)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, intercept_w + slope_w * x_line, color='crimson', linewidth=1.5)
ax.set_xlabel('Share managers, SOC group 1 (%)')
ax.set_ylabel('Bias ($e_i$)')
ax.set_title(f'Bias vs manager share (r = {r_w:.3f}, p = {p_w:.2e})')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots of all 4 covariates: bias vs each predictor
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
reg_vars = [
    ('pop_census', 'Usual resident population'),
    ('pop_density', 'Population density (residents per km²)'),
    ('young_pct', 'Share young (15-34)'),
    ('wealthier_pct', 'Share managers (SOC group 1)'),
]

for ax, (col, xlab) in zip(axes.ravel(), reg_vars):
    x = reg_df[col]
    y = reg_df['bias']
    ax.scatter(x, y, alpha=0.7, edgecolors='k', linewidths=0.3)
    slope, intercept, r, p, _ = linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, intercept + slope * x_line, color='crimson', linewidth=1.5)
    ax.set_xlabel(xlab)
    ax.set_ylabel('Bias ($e_i$)')
    ax.set_title(f'{xlab} vs bias (r = {r:.3f}, p = {p:.2e})')
plt.tight_layout()
plt.show()

In [ ]:
# Multivariate OLS
# 3. we combine all predictors, controlling for population size and density
model = smf.ols(
    'bias ~ pop_census + pop_density + young_pct + wealthier_pct',
    data=reg_df
).fit()
print(model.summary())

print('\n--- What do the results mean? ---')
for var, label in [
    ('pop_census', 'Usual resident population'),
    ('pop_density', 'Population density'),
    ('young_pct', 'Young population share'),
    ('wealthier_pct', 'Manager occupation share'),
]:
    coef = model.params[var]
    pval = model.pvalues[var]
    sig = 'significant' if pval < 0.05 else 'not significant'
    print(f'{label}: coefficient = {coef:.6f}, p = {pval:.3f} ({sig})')

print(
    f'\nR-squared = {model.rsquared:.3f} '
    f'({model.rsquared * 100:.1f}% of MSOA-level bias variation explained)'
)
